## C1 — Load Iris dataset and prepare the data

In [1]:
import os
import pandas as pd
import numpy as np

from IPython.display import display
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Load Iris dataset (equivalent to the Mage load_iris data loader).
iris = load_iris(as_frame=True)
df = iris.frame.copy()
df["target_name"] = df["target"].map(dict(enumerate(iris.target_names)))

print("DATASET INFORMATION")
print(f"Total rows: {len(df)}")
print(f"Feature columns: {iris.feature_names}")
print(f"Classes: {list(iris.target_names)}")
display(df.head())

DATASET INFORMATION
Total rows: 150
Feature columns: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Classes: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target,target_name
0,5.1,3.5,1.4,0.2,0,setosa
1,4.9,3.0,1.4,0.2,0,setosa
2,4.7,3.2,1.3,0.2,0,setosa
3,4.6,3.1,1.5,0.2,0,setosa
4,5.0,3.6,1.4,0.2,0,setosa


In [2]:
# Prepare an 80/20 train/test split.
X = df[iris.feature_names]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("DATA SPLIT")
print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")
print("Split ratio: 80/20")

DATA SPLIT
Training rows: 120
Testing rows: 30
Split ratio: 80/20


## C2 — Train and compare Decision Trees (max_depth 1–5)

In [3]:
# Test each max_depth using the same training and testing data.
depth_results = []

for depth in range(1, 6):
    candidate_model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )
    candidate_model.fit(X_train, y_train)

    candidate_predictions = candidate_model.predict(X_test)
    candidate_accuracy = accuracy_score(y_test, candidate_predictions)
    passed = candidate_accuracy >= 0.90

    depth_results.append({
        "max_depth": depth,
        "test_accuracy": candidate_accuracy,
        "status": "PASSED" if passed else "FAILED"
    })

    print("-" * 52)
    print(f"Max depth: {depth}")
    print(f"Test accuracy: {candidate_accuracy:.4f}")
    if passed:
        print("Quality test: PASSED")
    else:
        print("Quality test: FAILED (accuracy is below 0.90)")

depth_results_df = pd.DataFrame(depth_results)
print("\nDEPTH COMPARISON")
display(depth_results_df)

----------------------------------------------------
Max depth: 1
Test accuracy: 0.6667
Quality test: FAILED (accuracy is below 0.90)
----------------------------------------------------
Max depth: 2
Test accuracy: 0.9333
Quality test: PASSED
----------------------------------------------------
Max depth: 3
Test accuracy: 0.9667
Quality test: PASSED
----------------------------------------------------
Max depth: 4
Test accuracy: 0.9333
Quality test: PASSED
----------------------------------------------------
Max depth: 5
Test accuracy: 0.9333
Quality test: PASSED

DEPTH COMPARISON


,max_depth,test_accuracy,status
0,1,0.666667,FAILED
1,2,0.933333,PASSED
2,3,0.966667,PASSED
3,4,0.933333,PASSED
4,5,0.933333,PASSED


In [4]:
# Select the depth with the highest test accuracy.
best_result = depth_results_df.sort_values(
    by=["test_accuracy", "max_depth"],
    ascending=[False, True]
).iloc[0]

selected_depth = int(best_result["max_depth"])
print(f"Selected max_depth: {selected_depth}")
print(f"Selected test accuracy: {best_result['test_accuracy']:.4f}")

Selected max_depth: 3
Selected test accuracy: 0.9667


## C3 — Generate predictions, confusion matrix and classification metrics

This section computes the outputs directly from `y_test` and `y_pred`.

- `iris_predictions` is created from the model's actual predictions.
- `iris_metrics` is created by `classification_report(..., output_dict=True)`.
- The confusion matrix is calculated using `confusion_matrix(...)`.

No prediction records or metric values are manually entered.

In [5]:
# Train the final Decision Tree using the selected depth.
model = DecisionTreeClassifier(
    max_depth=selected_depth,
    random_state=42
)
model.fit(X_train, y_train)

# Generate predictions from the trained model.
y_pred = model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

print("FINAL MODEL")
print(f"Max depth: {selected_depth}")
print(f"Test accuracy: {test_accuracy:.4f}")

# Create prediction output automatically from actual and predicted labels.
predictions_df = pd.DataFrame({
    "actual": y_test.to_numpy(),
    "predicted": y_pred,
    "max_depth": selected_depth,
    "test_accuracy": test_accuracy
})

print("\nPREDICTION RESULTS")
print(f"Total predictions: {len(predictions_df)}")
print(f"Columns: {predictions_df.columns.tolist()}")
display(predictions_df)

FINAL MODEL
Max depth: 3
Test accuracy: 0.9667

PREDICTION RESULTS
Total predictions: 30
Columns: ['actual', 'predicted', 'max_depth', 'test_accuracy']


,actual,predicted,max_depth,test_accuracy
0,0,0,3,0.966667
1,2,2,3,0.966667
2,1,1,3,0.966667
3,1,1,3,0.966667
4,0,0,3,0.966667
5,1,1,3,0.966667
6,0,0,3,0.966667
7,0,0,3,0.966667
8,2,2,3,0.966667
9,1,1,3,0.966667


In [6]:
# Calculate and display the confusion matrix from the generated predictions.
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2])

print("CONFUSION MATRIX")
print("Rows = Actual; Columns = Predicted")
print(cm)

CONFUSION MATRIX
Rows = Actual; Columns = Predicted
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]


In [7]:
# Generate classification metrics automatically from actual and predicted labels.
report = classification_report(
    y_test,
    y_pred,
    labels=[0, 1, 2],
    target_names=list(iris.target_names),
    output_dict=True,
    zero_division=0
)

# Convert the calculated report into the requested CSV/table structure.
metrics_rows = []

for class_name in iris.target_names:
    class_metrics = report[class_name]
    metrics_rows.append({
        "class": class_name,
        "precision": class_metrics["precision"],
        "recall": class_metrics["recall"],
        "f1-score": class_metrics["f1-score"],
        "support": class_metrics["support"]
    })

# Add overall accuracy, macro average and weighted average rows.
metrics_rows.append({
    "class": "accuracy",
    "precision": report["accuracy"],
    "recall": report["accuracy"],
    "f1-score": report["accuracy"],
    "support": report["accuracy"]
})

for average_name in ["macro avg", "weighted avg"]:
    average_metrics = report[average_name]
    metrics_rows.append({
        "class": average_name,
        "precision": average_metrics["precision"],
        "recall": average_metrics["recall"],
        "f1-score": average_metrics["f1-score"],
        "support": average_metrics["support"]
    })

metrics_df = pd.DataFrame(
    metrics_rows,
    columns=["class", "precision", "recall", "f1-score", "support"]
)

print("CLASSIFICATION METRICS")
display(metrics_df.round(4))

CLASSIFICATION METRICS


,class,precision,recall,f1-score,support
0,setosa,1.0000,1.0000,1.0000,10.0000
1,versicolor,1.0000,0.9000,0.9474,10.0000
2,virginica,0.9091,1.0000,0.9524,10.0000
3,accuracy,0.9667,0.9667,0.9667,0.9667
4,macro avg,0.9697,0.9667,0.9666,30.0000
5,weighted avg,0.9697,0.9667,0.9666,30.0000


In [8]:
# Quality check for the final model.
assert test_accuracy >= 0.90, (
    f"Test accuracy {test_accuracy:.4f} is below the required 0.90"
)
assert len(predictions_df) == len(X_test)
assert len(metrics_df) == 6

print("Model quality test: PASSED")
print("Prediction output test: PASSED")
print("Metrics output test: PASSED")

Model quality test: PASSED
Prediction output test: PASSED
Metrics output test: PASSED


## C4 — Export CSV files and verify the saved outputs

In [9]:
# Export the generated prediction and metrics DataFrames.
predictions_file = "iris_predictions.csv"
metrics_file = "iris_metrics.csv"

predictions_df.to_csv(predictions_file, index=False)
metrics_df.to_csv(metrics_file, index=False)

print(f"Predictions saved to: {os.path.abspath(predictions_file)}")
print(f"Total predictions: {len(predictions_df)}")
print(f"Metrics saved to: {os.path.abspath(metrics_file)}")
print(f"Total metric rows: {len(metrics_df)}")

Predictions saved to: /home/oai/iris_predictions.csv
Total predictions: 30
Metrics saved to: /home/oai/iris_metrics.csv
Total metric rows: 6


In [10]:
# Read the exported CSV files back from disk and validate them.
saved_predictions = pd.read_csv(predictions_file)
saved_metrics = pd.read_csv(metrics_file)

assert os.path.exists(predictions_file)
assert os.path.exists(metrics_file)

assert saved_predictions.shape == (30, 4)
assert saved_metrics.shape == (6, 5)

assert list(saved_predictions.columns) == [
    "actual", "predicted", "max_depth", "test_accuracy"
]
assert list(saved_metrics.columns) == [
    "class", "precision", "recall", "f1-score", "support"
]

print("PREDICTION CSV VALIDATION: PASSED")
print(f"Rows: {len(saved_predictions)}")
display(saved_predictions.head())

print("\nMETRICS CSV VALIDATION: PASSED")
print(f"Rows: {len(saved_metrics)}")
display(saved_metrics)

PREDICTION CSV VALIDATION: PASSED
Rows: 30


,actual,predicted,max_depth,test_accuracy
0,0,0,3,0.966667
1,2,2,3,0.966667
2,1,1,3,0.966667
3,1,1,3,0.966667
4,0,0,3,0.966667



METRICS CSV VALIDATION: PASSED
Rows: 6


,class,precision,recall,f1-score,support
0,setosa,1.000000,1.000000,1.000000,10.000000
1,versicolor,1.000000,0.900000,0.947368,10.000000
2,virginica,0.909091,1.000000,0.952381,10.000000
3,accuracy,0.966667,0.966667,0.966667,0.966667
4,macro avg,0.969697,0.966667,0.966583,30.000000
5,weighted avg,0.969697,0.966667,0.966583,30.000000


In [11]:
print("PART C PIPELINE COMPLETED")
print(f"Selected max_depth: {selected_depth}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Prediction rows exported: {len(saved_predictions)}")
print(f"Metric rows exported: {len(saved_metrics)}")
print("All validation checks passed.")

PART C PIPELINE COMPLETED
Selected max_depth: 3
Test accuracy: 0.9667
Prediction rows exported: 30
Metric rows exported: 6
All validation checks passed.
